## EEG BrainLat — Pré-processamento e Extração de Features
### 2_Preprocessing

---

**Objetivo:** explorar os dados, pré-processar os sinais EEG e gerar o CSV com as features.

---

#### Estrutura do notebook

| Fase | Descrição |
|------|-----------|
| **0 — Configuração** | Imports globais e definições de caminhos |
| **2 — Exploração** | Demografia e metadados técnicos |
| **3 — Pré-processamento** | Filtragem, epoching e extração espectral |

---


---
### Phase 0 — Global Configuration

All imports and constants are centralised here.
**Execute this phase before any other.**


In [ ]:
# 0.1 -- Global Imports
import os
import re
import warnings
import traceback
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mne
from mne.time_frequency import psd_array_welch

from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc,
    recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.inspection import PartialDependenceDisplay
from xgboost import XGBClassifier
from scipy.stats import mannwhitneyu, spearmanr

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not installed. Run: pip install shap")

warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

SEED = 42
np.random.seed(SEED)
print("Imports loaded successfully.")


In [ ]:
# 0.2 -- Path Configuration and Constants
#
# Adapt the paths below to your environment.
# Only modify this cell; all subsequent cells use these variables.

ROOT_DIR      = "./Dataset_EEG_Alzheimer"
DIR_AD        = os.path.join(ROOT_DIR, "dataset_eeg_alzheimer")
DIR_HC        = os.path.join(ROOT_DIR, "dataset_eeg_hc")

# Synapse folder IDs (do not modify)
SYNAPSE_ID_AD = "syn53222482"
SYNAPSE_ID_HC = "syn53222486"

# Output files
OUT_CSV_FULL   = "eeg_features_brainlat_FULL.csv"
OUT_CSV_FAILED = "eeg_features_brainlat_failures.csv"

# Frequency bands (Hz)
BANDS = {
    "Delta": (0.5,  4.0),
    "Theta": (4.0,  8.0),
    "Alpha": (8.0, 13.0),
    "Beta" : (13.0, 30.0),
    "Gamma": (30.0, 45.0),
}
FMIN, FMAX = 0.5, 45.0

# Feature set used in the ML model
# Rel_Delta_mean is excluded: near-zero variance in this dataset (dead feature)
FEATURE_COLS = [
    "Rel_Theta_mean",
    "Rel_Alpha_mean",
    "Rel_Beta_mean",
    "Rel_Gamma_mean",
    "Razao_Theta_Alpha",
    "Razao_Theta_Beta",
    "Spectral_Entropy",
]

print("Configuration loaded.")
print(f"  AD directory : {DIR_AD}")
print(f"  HC directory : {DIR_HC}")
print(f"  Features     : {FEATURE_COLS}")


---
### Phase 2 -- Dataset Exploration

Before processing signals, the following are characterised:
- Number of participants per group.
- Demographic distribution (age, sex, country).
- Technical signal properties (duration, sampling rate, channel count).
- Visual inspection of a raw EEG trace.


In [ ]:
# 2.1 -- Demographic Analysis
path_demo_ad = os.path.join(DIR_AD, "demographics_ad_eeg_data.csv")
path_demo_hc = os.path.join(DIR_HC, "demographics_hc_eeg_data.csv")

try:
    df_ad_demo = pd.read_csv(path_demo_ad)
    df_hc_demo = pd.read_csv(path_demo_hc)
    df_ad_demo["group"] = "AD"
    df_hc_demo["group"] = "HC"
    df_demo = pd.concat([df_ad_demo, df_hc_demo], ignore_index=True)
    print(f"Loaded: {len(df_ad_demo)} AD | {len(df_hc_demo)} HC")
    print(f"Available columns: {list(df_demo.columns)}")
    display(df_demo.head(4))

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("BrainLat Dataset -- Demographic Profile", fontsize=13, fontweight="bold")

    counts_grp = df_demo["group"].value_counts()
    bars = axes[0].bar(counts_grp.index, counts_grp.values,
                       color=["#d62728", "#2ca02c"], edgecolor="white", lw=1.5)
    axes[0].bar_label(bars, fontsize=12, fontweight="bold", padding=4)
    axes[0].set_title("Participants per Group")
    axes[0].set_ylabel("N")
    axes[0].set_ylim(0, counts_grp.max() * 1.2)

    col_age = next((c for c in df_demo.columns if "age" in c.lower()), None)
    if col_age:
        for grp, col in [("AD", "#d62728"), ("HC", "#2ca02c")]:
            axes[1].hist(df_demo.loc[df_demo["group"] == grp, col_age].dropna(),
                         bins=12, alpha=0.7, color=col, label=grp)
        axes[1].set_title("Age Distribution")
        axes[1].set_xlabel("Age (years)")
        axes[1].set_ylabel("N")
        axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, "Age column\nnot found",
                     ha="center", va="center", transform=axes[1].transAxes)
        axes[1].set_axis_off()

    col_sex = next((c for c in df_demo.columns
                    if "sex" in c.lower() or "gen" in c.lower()), None)
    if col_sex:
        df_demo["sex_label"] = df_demo[col_sex].map({0: "Female", 1: "Male"}).fillna(
            df_demo[col_sex].astype(str))
        tab = pd.crosstab(df_demo["group"], df_demo["sex_label"])
        for cat in ["Female", "Male"]:
            if cat not in tab.columns:
                tab[cat] = 0
        tab = tab[["Female", "Male"]]
        x = np.arange(len(tab))
        w = 0.35
        color_map = {"Female": "#ff7f0e", "Male": "#1f77b4"}
        for i, sex in enumerate(["Female", "Male"]):
            b = axes[2].bar(x + (i - 0.5) * w, tab[sex].values, width=w,
                            color=color_map[sex], alpha=0.8, label=sex)
            axes[2].bar_label(b, fontsize=9, padding=2)
        axes[2].set_xticks(x)
        axes[2].set_xticklabels(tab.index.tolist())
        axes[2].set_title("Sex Distribution")
        axes[2].set_ylabel("N")
        axes[2].legend()
        axes[2].set_ylim(0, tab.values.max() * 1.2)
    else:
        axes[2].text(0.5, 0.5, "Sex column\nnot found",
                     ha="center", va="center", transform=axes[2].transAxes)
        axes[2].set_axis_off()

    plt.tight_layout()
    plt.show()

except FileNotFoundError as e:
    print(f"Demographic CSV not found: {e}")
    print("Verify DIR_AD and DIR_HC paths.")


In [ ]:
# 2.2 -- Technical Metadata of EEG Signals
print("Scanning EEG file metadata...\n")

tech_records = []
for group, folder in [("AD", DIR_AD), ("HC", DIR_HC)]:
    for fpath in Path(folder).rglob("*.set"):
        parts_upper = {p.upper() for p in fpath.parts}
        country = ("Argentina" if "AR" in parts_upper
                   else "Chile" if "CL" in parts_upper else "Other")
        try:
            raw = mne.io.read_raw_eeglab(str(fpath), preload=False, verbose=False)
            ch_types = Counter(raw.get_channel_types())
            tech_records.append({
                "Group"       : group,
                "Country"     : country,
                "File"        : fpath.name,
                "Duration_s"  : round(raw.times[-1], 2),
                "Fs_Hz"       : raw.info["sfreq"],
                "N_Channels"  : raw.info["nchan"],
                "N_EEG"       : ch_types.get("eeg", 0),
                "N_Bad"       : len(raw.info["bads"]),
            })
        except Exception:
            tech_records.append({
                "Group": group, "Country": country, "File": fpath.name,
                "Duration_s": None, "Fs_Hz": None,
                "N_Channels": None, "N_EEG": None, "N_Bad": None,
            })

df_tech = pd.DataFrame(tech_records)
print(f"{len(df_tech)} files scanned.")
display(df_tech.head(8))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("EEG Signal Technical Characteristics", fontsize=13, fontweight="bold")

for grp, col in [("AD", "#d62728"), ("HC", "#2ca02c")]:
    sub = df_tech.loc[df_tech["Group"] == grp, "Duration_s"].dropna()
    axes[0].hist(sub, bins=12, alpha=0.7, color=col, label=grp)
axes[0].set_title("Recording Duration")
axes[0].set_xlabel("Duration (s)")
axes[0].set_ylabel("N")
axes[0].legend()

fs_counts = df_tech["Fs_Hz"].value_counts()
axes[1].bar(fs_counts.index.astype(str), fs_counts.values, color="#1f77b4")
axes[1].set_title("Sampling Rate")
axes[1].set_xlabel("Fs (Hz)")
axes[1].set_ylabel("N files")

ch_counts = df_tech["N_Channels"].value_counts().sort_index()
axes[2].bar(ch_counts.index.astype(str), ch_counts.values, color="#9467bd")
axes[2].set_title("Channel Count")
axes[2].set_xlabel("N Channels")
axes[2].set_ylabel("N files")

plt.tight_layout()
plt.show()


In [ ]:
# 2.3 -- Raw EEG Visual Inspection
# Opens the first .set file found and plots the first 10 s of 5 representative channels.

first_set = next(Path(DIR_AD).rglob("*.set"), None) or next(Path(DIR_HC).rglob("*.set"), None)

if first_set:
    print(f"Visualising: {first_set.name}")
    raw_viz  = mne.io.read_raw_eeglab(str(first_set), preload=True, verbose=False)
    sfreq    = raw_viz.info["sfreq"]
    channels = [c for c in raw_viz.ch_names
                if raw_viz.get_channel_types([c])[0] == "eeg"][:5]
    t_max    = min(10.0, raw_viz.times[-1])
    n_pts    = int(t_max * sfreq)
    times    = raw_viz.times[:n_pts]
    data     = raw_viz.get_data(picks=channels)[:, :n_pts]

    fig, axes = plt.subplots(len(channels), 1, figsize=(14, 7), sharex=True)
    fig.suptitle(f"Raw EEG -- {first_set.name} (first {t_max:.0f} s)",
                 fontsize=12, fontweight="bold")
    for i, (ch, ax) in enumerate(zip(channels, axes)):
        ax.plot(times, data[i] * 1e6, lw=0.7, color=plt.cm.tab10(i))
        ax.set_ylabel(f"{ch}\n(uV)", fontsize=8)
        ax.yaxis.set_tick_params(labelsize=7)
    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()
else:
    print("No .set file found. Run Phase 1 first.")


---
### Phase 3 -- Preprocessing and Feature Extraction

#### Processing Pipeline

```
.set (EEGLAB)
   -> Robust loading (.set + optional .fdt)
   -> Re-referencing (global average)
   -> Band-pass filter (0.5 - 45 Hz, FIR firwin)
   -> EEG channel selection
   -> Fixed-length epochs (4 s)
   -> Per-subject amplitude normalisation (global RMS = 1)
   -> Welch PSD (per epoch, n_fft = min(512, epoch_length))
   -> Features: relative band power + spectral ratios + spectral entropy
   -> Output: CSV with all epochs from all subjects
```

#### Extracted Features

| Feature | Description | Clinical relevance |
|---------|-------------|-------------------|
| `Rel_Theta_mean` | Relative Theta power (4-8 Hz) | Elevated in AD: cortical slowing |
| `Rel_Alpha_mean` | Relative Alpha power (8-13 Hz) | Reduced in AD: thalamocortical dysfunction |
| `Rel_Beta_mean` | Relative Beta power (13-30 Hz) | Variable in AD |
| `Rel_Gamma_mean` | Relative Gamma power (30-45 Hz) | Reduced in AD: synchronisation deficit |
| `Razao_Theta_Alpha` | Theta / Alpha ratio | Classic marker of cognitive decline |
| `Razao_Theta_Beta` | Theta / Beta ratio | Related to generalised EEG slowing |
| `Spectral_Entropy` | Spectral entropy | Reduced in AD: pathological EEG regularity |


In [ ]:
# 3.1 -- Auxiliary Functions: Subject ID and File Loading

def extract_subject_id(fpath: Path) -> str:
    """Extract subject ID (e.g., sub-30007) from the file name."""
    m = re.search(r"(sub-\d+)", fpath.stem, flags=re.IGNORECASE)
    if m:
        return m.group(1)
    m = re.search(r"sub[-_]?(\d+)", fpath.stem, flags=re.IGNORECASE)
    if m:
        return f"sub-{m.group(1)}"
    return fpath.stem


def infer_country(fpath: Path) -> str:
    """Infer recording country from the folder path (AR=Argentina, CL=Chile)."""
    parts = {p.upper() for p in fpath.parts}
    if "AR" in parts:
        return "Argentina"
    if "CL" in parts:
        return "Chile"
    return "Other"


def load_raw_safe(fpath: Path):
    """
    Load a .set file robustly.
    If the paired .fdt file is absent from the working directory,
    attempt a temporary-directory copy as a fallback.
    """
    try:
        return mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=False)
    except Exception as exc:
        msg = str(exc).lower()
        if "fdt" in msg or "not found" in msg:
            fdt = fpath.with_suffix(".fdt")
            if fdt.exists():
                from tempfile import TemporaryDirectory
                with TemporaryDirectory() as tmp:
                    tmp_p = Path(tmp)
                    (tmp_p / fpath.name).write_bytes(fpath.read_bytes())
                    (tmp_p / fdt.name).write_bytes(fdt.read_bytes())
                    return mne.io.read_raw_eeglab(
                        str(tmp_p / fpath.name), preload=True, verbose=False)
        raise exc

print("Auxiliary functions defined.")


In [ ]:
# 3.2 -- Spectral Feature Extraction Function (Welch PSD)

def extract_eeg_features(data: np.ndarray, sfreq: float) -> dict:
    """
    Compute spectral features for ONE EEG epoch.

    Parameters
    ----------
    data  : 2D array (n_channels x n_samples), amplitude-normalised (global RMS ~ 1)
    sfreq : sampling frequency (Hz)

    Returns
    -------
    dict with relative band powers, spectral ratios, and spectral entropy.
    """
    data = np.asarray(data, dtype=float)
    if data.ndim != 2:
        raise ValueError(f"Expected 2D array, got shape={data.shape}")

    n_fft = min(512, data.shape[-1])
    psds, freqs = psd_array_welch(
        data, sfreq=sfreq, fmin=FMIN, fmax=FMAX,
        n_fft=n_fft, verbose=False
    )
    psd_mean = np.mean(psds, axis=0)

    integrate = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

    band_power = {}
    for band, (flo, fhi) in BANDS.items():
        idx = (freqs >= flo) & (freqs < fhi)
        band_power[band] = (float(integrate(psd_mean[idx], freqs[idx]))
                            if np.sum(idx) >= 2 else 0.0)

    total = sum(band_power.values()) + 1e-12

    features = {
        "Rel_Delta_mean" : band_power["Delta"] / total,
        "Rel_Theta_mean" : band_power["Theta"] / total,
        "Rel_Alpha_mean" : band_power["Alpha"] / total,
        "Rel_Beta_mean"  : band_power["Beta"]  / total,
        "Rel_Gamma_mean" : band_power["Gamma"] / total,
    }
    features["Razao_Theta_Alpha"] = (features["Rel_Theta_mean"]
                                     / (features["Rel_Alpha_mean"] + 1e-12))
    features["Razao_Theta_Beta"]  = (features["Rel_Theta_mean"]
                                     / (features["Rel_Beta_mean"]  + 1e-12))

    psd_norm = psd_mean / (np.sum(psd_mean) + 1e-12)
    features["Spectral_Entropy"] = float(-np.sum(psd_norm * np.log2(psd_norm + 1e-12)))

    return features

print("Feature extraction function defined.")


In [ ]:
# 3.3 -- Main Preprocessing and Extraction Loop
#
# For each .set file:
#   1. Re-reference to global average
#   2. Band-pass filter (0.5 - 45 Hz)
#   3. Retain EEG channels only
#   4. Create fixed-length 4 s epochs
#   5. Per-subject amplitude normalisation (global RMS = 1)
#   6. Extract spectral features per epoch

all_epochs   = []
failed_files = []

for group, folder in [("AD", DIR_AD), ("HC", DIR_HC)]:
    files = list(Path(folder).rglob("*.set"))
    print(f"\n{'=' * 55}")
    print(f"  Group {group}: {len(files)} file(s) found")
    print(f"{'=' * 55}")

    for fpath in files:
        try:
            subject_id = extract_subject_id(fpath)
            country    = infer_country(fpath)
            raw        = load_raw_safe(fpath)

            raw.set_eeg_reference("average", projection=False)
            raw.filter(FMIN, FMAX, fir_design="firwin", verbose=False)
            raw.pick_types(eeg=True, eog=False, ecg=False,
                           stim=False, emg=False, misc=False)

            epochs = mne.make_fixed_length_epochs(
                raw, duration=4.0, preload=True, verbose=False)
            if len(epochs) == 0:
                raise RuntimeError("No epochs generated (recording too short?).")

            data  = epochs.get_data()
            sfreq = float(raw.info["sfreq"])

            global_rms = np.sqrt(np.mean(data.reshape(-1, data.shape[-1]) ** 2))
            if global_rms < 1e-12:
                global_rms = 1.0
            data_norm = data / global_rms

            for i, epoch in enumerate(data_norm):
                feat = extract_eeg_features(epoch, sfreq)
                feat.update({
                    "subject_id"  : subject_id,
                    "label"       : group,
                    "country"     : country,
                    "epoch_id"    : i,
                    "source_file" : fpath.name,
                })
                all_epochs.append(feat)

            print(f"  OK   {subject_id:<18} | epochs={len(data):3d} "
                  f"| RMS={global_rms:.4e} | country={country}")

        except Exception as exc:
            failed_files.append({
                "group"     : group,
                "file"      : fpath.name,
                "path"      : str(fpath),
                "error"     : str(exc),
                "traceback" : traceback.format_exc(),
            })
            print(f"  FAIL {fpath.name}  ->  {exc}")


#### Note: Why Some `.set` Files Fail to Load

EEGLAB `.set` files store only metadata. The raw EEG binary data reside in a paired
`.fdt` file with the same base name.

**Error:** `File ...sub-XXXXX_rs_eeg.fdt not found`

This occurs when the `.fdt` partner is absent from the same directory.

**Impact:** Subjects without the `.fdt` file cannot be processed.
All AD subjects in BrainLat are self-contained (data embedded in `.set`);
a subset of HC subjects use the split `.set`/`.fdt` format.

**Action:** Verify whether missing `.fdt` files can be recovered from the original download.
The pipeline continues with available data.


In [ ]:
# 3.4 -- Save Extracted Features and Print Extraction Summary

df_full   = pd.DataFrame(all_epochs)
df_failed = pd.DataFrame(failed_files)

if df_full.empty:
    raise RuntimeError("No epochs extracted. Check paths and Phase 1 download.")

cols_meta = ["subject_id", "label", "country", "epoch_id", "source_file"]
cols_feat = [c for c in df_full.columns if c not in cols_meta]
df_full   = df_full[cols_meta + cols_feat]

df_full.to_csv(OUT_CSV_FULL,   index=False, encoding="utf-8-sig")
df_failed.to_csv(OUT_CSV_FAILED, index=False, encoding="utf-8-sig")

n_subj_ad = df_full.loc[df_full["label"] == "AD", "subject_id"].nunique()
n_subj_hc = df_full.loc[df_full["label"] == "HC", "subject_id"].nunique()
n_ep_ad   = (df_full["label"] == "AD").sum()
n_ep_hc   = (df_full["label"] == "HC").sum()

print(f"{'=' * 55}")
print("  EXTRACTION SUMMARY")
print(f"{'=' * 55}")
print(f"  AD subjects  : {n_subj_ad}")
print(f"  HC subjects  : {n_subj_hc}")
print(f"  AD epochs    : {n_ep_ad:,}")
print(f"  HC epochs    : {n_ep_hc:,}")
print(f"  Total epochs : {len(df_full):,}")
print(f"  Failures     : {len(df_failed)}")
print(f"{'=' * 55}")
print(f"\n  Saved to: {OUT_CSV_FULL}")

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("Spectral Feature Distributions by Group", fontsize=13, fontweight="bold")
for i, feat in enumerate(cols_feat):
    ax = axes[i // 4][i % 4]
    for grp, col in [("AD", "#d62728"), ("HC", "#2ca02c")]:
        sub = df_full.loc[df_full["label"] == grp, feat].dropna()
        ax.hist(sub, bins=30, alpha=0.6, color=col, label=grp, density=True)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel("Feature value", fontsize=8)
    ax.set_ylabel("Density",       fontsize=8)
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=8)
for j in range(len(cols_feat), 8):
    axes[j // 4][j % 4].set_visible(False)
plt.tight_layout()
plt.show()
